In [ ]:
%pip install -q sentence-transformers transformers pillow sqlalchemy torch numpy pandas chromadb

# Kernel restart wymagany po instalacji

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Inicjalizacja i Konfiguracja

In [ ]:
import os
# Wyłącz TensorFlow (używamy tylko PyTorch)
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import torch
import numpy as np
from pathlib import Path
from typing import List, Optional, Dict
from dataclasses import dataclass
from datetime import datetime

# Importy baz danych i ML
from sqlalchemy import create_engine, Column, Integer, String, Float, JSON, ForeignKey
from sqlalchemy.orm import declarative_base, sessionmaker, Session, relationship
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
import chromadb

# Konfiguracja modeli i baz danych
EMBEDDING_MODEL = "clip-ViT-B-32"  # CLIP do obrazów i tekstu
LLM_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # mały model językowy
DATABASE_URL = "sqlite:///metal_parts.db"  # SQLite dla danych
CHROMA_PATH = "./chroma_db"  # ChromaDB dla wektorów
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # GPU jak jest
USE_CHROMA = True  # używać ChromaDB (szybciej)

print(f"✓ Urządzenie: {DEVICE}")
print(f"✓ Baza SQLite: {DATABASE_URL}")
print(f"✓ Chroma ścieżka: {CHROMA_PATH}")

# SQLAlchemy setup
Base = declarative_base()
engine = create_engine(DATABASE_URL, echo=False)
SessionLocal = sessionmaker(bind=engine)

# ChromaDB init
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
COLLECTION_NAME = "metal_parts_images"
try:
    chroma_collection = chroma_client.get_collection(name=COLLECTION_NAME)
    print(f"✓ Chroma kolekcja '{COLLECTION_NAME}' załadowana ({chroma_collection.count()} rekordów)")
except:
    chroma_collection = chroma_client.create_collection(name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"})
    print(f"✓ Nowa Chroma kolekcja '{COLLECTION_NAME}'")

c:\Users\hubik\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Urządzenie: cpu
✓ Baza danych SQLite: sqlite:///metal_parts.db
✓ Chroma DB ścieżka: ./chroma_db
✓ Kolekcja Chroma 'metal_parts_images' załadowana (liczba rekordów: 7)


## 2. Modele Danych

In [ ]:
@dataclass
class MetalPart:
    """Część metalowa w pamięci"""
    part_id: str
    description: str
    material: str
    category: str  # np. "łożyska", "śruby"
    dimensions: Dict  # np. {"średnica": 8, "długość": 20}
    tags: List[str]  # tagi do szukania
    image_path: Optional[str] = None  # ścieżka do obrazka
    image_embedding: Optional[List[float]] = None  # wektor obrazka
    text_embedding: Optional[List[float]] = None  # wektor tekstu


class PartDB(Base):
    """Tabela części w bazie"""
    __tablename__ = "metal_parts"

    id = Column(Integer, primary_key=True, autoincrement=True)
    part_id = Column(String, unique=True, nullable=False)  # ID części
    description = Column(String, nullable=False)  # opis
    material = Column(String, nullable=True)  # materiał
    category = Column(String, nullable=True)  # kategoria
    dimensions = Column(JSON, nullable=True)  # wymiary jako JSON
    tags = Column(JSON, nullable=True)  # tagi jako lista
    image_path = Column(String, nullable=True)  # ścieżka obrazka
    image_embedding = Column(String, nullable=True)  # embedding obrazka
    text_embedding = Column(String, nullable=True)  # embedding tekstu
    created_at = Column(String, nullable=True)  # kiedy dodano


class SearchLog(Base):
    """Log wyszukiwań"""
    __tablename__ = "search_logs"

    id = Column(Integer, primary_key=True, autoincrement=True)
    query = Column(String, nullable=False)  # co szukano
    query_type = Column(String, nullable=False)  # "text", "image"
    top_k = Column(Integer, nullable=True)  # ile wyników
    results_count = Column(Integer, nullable=True)  # ile znalazł
    timestamp = Column(String, nullable=True)  # kiedy


def init_database():
    """Stwórz tabele jak nie ma"""
    Base.metadata.create_all(engine)
    print("✓ Baza gotowa")

init_database()

✓ Baza danych zainicjalizowana


## 3. Embeddingi i Ekstrakcja Cech

In [ ]:
# Ładowanie modelu do tworzenia embeddingów (CLIP - obraz + tekst)
print("Ładowanie modelu embeddingów...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
print(f"✓ Model embeddingów załadowany: {EMBEDDING_MODEL}")

def get_image_embedding(image_path: str) -> List[float]:
    """Zrób wektor z obrazka używając CLIP"""
    from PIL import Image
    try:
        img = Image.open(image_path).convert('RGB')  # otwórz obrazek
        img_embedding = embedding_model.encode(img, normalize_embeddings=True)  # wektor
        return img_embedding.tolist()  # jako lista
    except Exception as e:
        print(f"✗ Błąd obrazka {image_path}: {e}")
        return None

Ładowanie modelu embeddingów...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✓ Model embeddingów załadowany: clip-ViT-B-32


## 4. Funkcje do Indeksowania Części

In [ ]:
def add_part_to_db(db: Session, part: MetalPart) -> bool:
    """Dodaj część do baz (Chroma + SQLite)"""
    try:
        # Zrób embedding obrazka
        image_emb = None
        if part.image_path and Path(part.image_path).exists():
            image_emb = get_image_embedding(part.image_path)
        else:
            print(f"✗ Brak obrazka: {part.image_path}")
            return False

        # Embedding na string dla SQLite
        image_emb_str = ";".join(str(x) for x in image_emb) if image_emb else None

        # Dodaj do ChromaDB (szybkie szukanie)
        if USE_CHROMA and image_emb:
            chroma_collection.add(
                ids=[part.part_id],  # ID części
                embeddings=[image_emb],  # wektor obrazka
                documents=[part.description],  # tekst do szukania
                metadatas=[{  # metadane
                    "part_id": part.part_id,
                    "description": part.description,
                    "material": part.material,
                    "category": part.category,
                    "image_path": part.image_path,
                    "tags": ",".join(part.tags) if part.tags else ""
                }]
            )

        # Dodaj do SQLite (backup i dane)
        db_part = PartDB(
            part_id=part.part_id,
            description=part.description,
            material=part.material,
            category=part.category,
            dimensions=part.dimensions,
            tags=part.tags,
            image_path=part.image_path,
            text_embedding=None,  # nie używamy
            image_embedding=image_emb_str,  # embedding jako string
            created_at=datetime.now().isoformat()
        )
        db.add(db_part)
        db.commit()
        return True
    except Exception as e:
        print(f"✗ Błąd dodawania: {e}")
        return False


def load_parts_from_db(db: Session) -> List[PartDB]:
    """Załaduj wszystkie części z SQLite"""
    return db.query(PartDB).all()


def parse_embedding_from_db(emb_str: str) -> List[float]:
    """String z bazy na listę floatów"""
    if not emb_str:
        return None
    return [float(x) for x in emb_str.split(";")]


def cosine_similarity(a: List[float], b: List[float]) -> float:
    """Podobieństwo kosinusowe między wektorami (0-1)"""
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

print("✓ Funkcje indeksowania (image-only + Chroma) załadowane")

✓ Funkcje indeksowania (image-only + Chroma) załadowane


## 5. Retriever - Wyszukiwanie Podobnych Części

In [ ]:
def search_parts_by_image(db: Session, image_path: str, top_k: int = 5, category_filter: Optional[str] = None) -> List[tuple]:
    """Szukaj podobnych części po obrazku"""
    # Zrób embedding zapytania
    query_emb = get_image_embedding(image_path)
    if not query_emb:
        return []

    # Najpierw spróbuj ChromaDB (szybciej)
    if USE_CHROMA and chroma_collection.count() > 0:
        # Filtr kategorii jak podano
        where_filter = None
        if category_filter:
            where_filter = {"category": {"$eq": category_filter}}

        # Szukaj w ChromaDB
        results = chroma_collection.query(
            query_embeddings=[query_emb],  # wektor zapytania
            n_results=top_k,  # ile wyników
            where=where_filter,  # filtr
            include=["embeddings", "documents", "metadatas", "distances"]  # co zwrócić
        )

        if not results["ids"] or not results["ids"][0]:
            return []

        # Przetwórz wyniki na nasz format
        part_results = []
        for idx, (doc_id, metadata, distance) in enumerate(
            zip(results["ids"][0], results["metadatas"][0], results["distances"][0])
        ):
            # Odległość na podobieństwo
            similarity_score = 1 - distance
            
            # Zrób obiekt jak PartDB
            part_proxy = type('PartDB', (), {
                'part_id': metadata.get('part_id'),
                'description': metadata.get('description'),
                'material': metadata.get('material'),
                'category': metadata.get('category'),
                'image_path': metadata.get('image_path'),
                'tags': metadata.get('tags', '').split(',') if metadata.get('tags') else [],
                'dimensions': {}
            })()
            part_results.append((part_proxy, similarity_score))

        return part_results

    # Jak nie ma Chroma, szukaj w SQLite
    print("⚠ Fallback na SQLite")
    all_parts = load_parts_from_db(db)
    results = []
    for part in all_parts:
        # Filtruj kategorię
        if category_filter and part.category != category_filter:
            continue
        # Pobierz embedding części
        part_emb = parse_embedding_from_db(part.image_embedding)
        if not part_emb:
            continue
        # Oblicz podobieństwo
        score = cosine_similarity(query_emb, part_emb)
        results.append((part, score))

    # Sortuj i zwróć top_k
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]

print("✓ Retriever (Chroma + fallback SQLite) załadowany")

✓ Retriever (Chroma + fallback SQLite) załadowany


In [ ]:
from pathlib import Path

def index_images_from_folder(db: Session, root: Path = Path("obrazy/metal"), default_category: str = "metal") -> int:
    """Zindeksuj obrazy z folderu jako części.
    
    Co robi:
    1. Przeszukuje folder rekurencyjnie
    2. Dla każdego obrazka tworzy MetalPart
    3. Wywołuje add_part_to_db() - robi embedding, zapisuje do baz
    4. Zwraca liczbę dodanych
    
    Args:
    - db: sesja bazy
    - root: folder z obrazkami
    - default_category: domyślna kategoria
    
    Returns: liczba zaindeksowanych
    """
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    count = 0
    for p in root.rglob("*"):
        if p.suffix.lower() in exts:
            part = MetalPart(
                part_id=p.stem,
                description=p.stem.replace("_", " ").replace("-", " "),
                material="N/A",
                category=default_category,
                dimensions={},
                tags=[default_category],
                image_path=str(p.as_posix()),
            )
            if add_part_to_db(db, part):
                count += 1
    print(f"✓ Zaindeksowano {count} obrazów z: {root}")
    return count

## 6. LLM - Generacja Raportu

In [ ]:
print("Ładowanie modelu LLM...")
try:
    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
    llm_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, low_cpu_mem_usage=True)
    llm_model.to(DEVICE)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print(f"✓ LLM załadowany: {LLM_MODEL}")
except Exception as e:
    print(f"✗ Błąd ładowania LLM: {e}")
    llm_model = None
    tokenizer = None


def generate_report(results: List[tuple], query: str) -> str:
    """Raport z LLM (TinyLlama).
    
    Co robi:
    1. Sprawdza czy LLM jest
    2. Robi kontekst z wyników
    3. Buduje prompt dla LLM
    4. Generuje odpowiedź
    5. Czyści duplikaty
    
    Jak nie ma LLM, używa prostego raportu.
    
    Args:
        results: wyniki wyszukiwania
        query: co szukano
    
    Returns:
        raport tekstowy
    """
    if not llm_model or not tokenizer:
        return generate_simple_report(results, query)
    
    context = "\n\n".join([
        f"Część {i+1}:\n"
        f"  ID: {part.part_id}\n"
        f"  Opis: {part.description}\n"
        f"  Materiał: {part.material}\n"
        f"  Kategoria: {part.category}\n"
        f"  Wymiary: {part.dimensions}\n"
        f"  Tagi: {', '.join(part.tags or [])}\n"
        f"  Dopasowanie: {score*100:.1f}%"
        for i, (part, score) in enumerate(results)
    ])
    
    prompt = f"""
    Użytkownik szuka części metalowych. Zapytanie: "{query}"
    
    Znalezione części:
    {context}
    
    Na podstawie wyników wyszukiwania, wygeneruj krótki, rzeczowy raport o znalezionych częściach.
    Wskaż, które części najlepiej pasują do zapytania i dlaczego.
    """
    
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(DEVICE)
    
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("Na podstawie wyników")[-1].strip() if "Na podstawie wyników" in response else response


def generate_simple_report(results: List[tuple], query: str) -> str:
    """Prosty raport bez LLM.
    
    Robi tekst z:
    - nagłówkiem
    - info o każdej części
    - procent dopasowania
    - wszystkie metadane
    
    Używany jak nie ma LLM lub trzeba szybko.
    
    Args:
        results: wyniki
        query: zapytanie
    
    Returns:
        tekst raportu
    """
    report = f"RAPORT WYSZUKIWANIA\n"
    report += f"Zapytanie: {query}\n"
    report += f"Znaleziono: {len(results)} części\n"
    report += "\n" + "="*60 + "\n\n"
    
    for i, (part, score) in enumerate(results, 1):
        report += f"#{i} (Dopasowanie: {score*100:.1f}%)\n"
        report += f"  ID: {part.part_id}\n"
        report += f"  Opis: {part.description}\n"
        report += f"  Materiał: {part.material}\n"
        report += f"  Kategoria: {part.category}\n"
        if part.dimensions:
            report += f"  Wymiary: {part.dimensions}\n"
        if part.tags:
            report += f"  Tagi: {', '.join(part.tags)}\n"
        report += "\n"
    
    return report

print("✓ Funkcje generacji raportu załadowane")

Ładowanie modelu LLM...
✓ LLM załadowany: TinyLlama/TinyLlama-1.1B-Chat-v1.0
✓ Funkcje generacji raportu załadowane
✓ LLM załadowany: TinyLlama/TinyLlama-1.1B-Chat-v1.0
✓ Funkcje generacji raportu załadowane


## 7. Główna Funkcja RAG

In [ ]:
def rag_search_images(image_path: str, category_filter: Optional[str] = None, top_k: int = 5) -> Dict:
    """
    Główna funkcja RAG - szukaj części po obrazku.
    
    Łączy wszystko:
    1. RETRIEVAL: Znajdź podobne części przez search_parts_by_image()
       - embedding obrazka
       - szukaj w ChromaDB lub SQLite
       - top_k wyników
    
    2. AUGMENTATION: Zamień na słowniki
       - obiekty na JSON
       - wszystkie metadane
    
    3. GENERATION: Zrób raport
       - generate_simple_report()
       - może LLM później
    
    Loguje każde wyszukiwanie.
    
    Args:
        image_path: obrazek do szukania
        category_filter: filtr kategorii
        top_k: ile wyników
    
    Returns:
        dict z wynikami, raportem, czasem
    """
    with SessionLocal() as db:
        results = search_parts_by_image(db, image_path, top_k=top_k, category_filter=category_filter)

        results_dicts = [
            ({
                "part_id": part.part_id,
                "description": part.description,
                "material": part.material,
                "category": part.category,
                "dimensions": part.dimensions,
                "tags": part.tags,
                "image_path": part.image_path,
            }, score)
            for part, score in results
        ]

        report = generate_simple_report(results, Path(image_path).name)

        log = SearchLog(
            query=Path(image_path).name,
            query_type="image",
            top_k=top_k,
            results_count=len(results),
            timestamp=datetime.now().isoformat()
        )
        db.add(log)
        db.commit()

        return {
            "query_image": image_path,
            "results": results_dicts,
            "report": report,
            "timestamp": datetime.now().isoformat()
        }

print("✓ Funkcja wyszukiwania (image-only) załadowana")

✓ Funkcja wyszukiwania (image-only) załadowana


## 8. Przykładowe Dane i Test

In [ ]:
from pathlib import Path

with SessionLocal() as db:
    # Wyczyść stare dane
    db.query(PartDB).delete()
    db.commit()
    
    # Wyczyść Chroma jak używasz
    if USE_CHROMA:
        try:
            chroma_client.delete_collection(name=COLLECTION_NAME)
            globals()['chroma_collection'] = chroma_client.create_collection(name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"})
            print(f"✓ Wyczyszczono Chroma")
        except Exception as e:
            print(f"⚠ Błąd: {e}")

    # Dodaj obrazy z folderu jako testowe dane
    index_images_from_folder(db, Path("obrazy/metal"), default_category="metal")
    
    # Statystyki
    if USE_CHROMA:
        print(f"✓ Chroma: {chroma_collection.count()} rekordów")

✓ Wyczyszczono kolekcję Chroma 'metal_parts_images'
✓ Zaindeksowano 7 obrazów z: obrazy\metal
✓ Chroma DB zawiera 7 rekordów
✓ Zaindeksowano 7 obrazów z: obrazy\metal
✓ Chroma DB zawiera 7 rekordów


## 9. Testy Wyszukiwania RAG

In [ ]:
print("\n" + "="*70)
print("TEST 1: Wyszukiwanie obrazem")
print("="*70)

from pathlib import Path
exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
candidates = [p for p in Path("obrazy/metal").rglob("*") if p.suffix.lower() in exts]

if not candidates:
    print("Brak obrazów")
else:
    # Pierwszy obrazek jako zapytanie
    query_img = str(candidates[0])
    print(f"Obraz: {query_img}")
    # Szukaj bez filtra
    result1 = rag_search_images(query_img, top_k=5)
    print(f"\n{result1['report']}")


TEST 1: Image-only search (obrazy/metal)
Zapytanie (obraz): obrazy\metal\m1.jpg

Raport:
RAPORT WYSZUKIWANIA
Zapytanie: m1.jpg
Znaleziono: 5 części


#1 (Dopasowanie: 100.0%)
  ID: m1
  Opis: m1
  Materiał: N/A
  Kategoria: metal
  Tagi: metal

#2 (Dopasowanie: 81.9%)
  ID: m7
  Opis: m7
  Materiał: N/A
  Kategoria: metal
  Tagi: metal

#3 (Dopasowanie: 81.6%)
  ID: m2
  Opis: m2
  Materiał: N/A
  Kategoria: metal
  Tagi: metal

#4 (Dopasowanie: 80.8%)
  ID: m6
  Opis: m6
  Materiał: N/A
  Kategoria: metal
  Tagi: metal

#5 (Dopasowanie: 80.5%)
  ID: m4
  Opis: m4
  Materiał: N/A
  Kategoria: metal
  Tagi: metal




In [ ]:
print("\n" + "="*70)
print("TEST 2: Wyszukiwanie z filtrem kategorii")
print("="*70)

from pathlib import Path
exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
candidates = [p for p in Path("obrazy/metal").rglob("*") if p.suffix.lower() in exts]

if len(candidates) < 2:
    print("Za mało obrazów")
else:
    # Drugi obrazek
    query_img = str(candidates[1])
    print(f"Obraz: {query_img}")
    # Szukaj z filtrem kategorii
    result2 = rag_search_images(query_img, category_filter="metal", top_k=5)
    print(f"\n{result2['report']}")


TEST 2: Image-only z filtrem kategorii (metal)
Zapytanie (obraz): obrazy\metal\m2.webp

Raport:
RAPORT WYSZUKIWANIA
Zapytanie: m2.webp
Znaleziono: 5 części


#1 (Dopasowanie: 100.0%)
  ID: m2
  Opis: m2
  Materiał: N/A
  Kategoria: metal
  Tagi: metal

#2 (Dopasowanie: 86.5%)
  ID: m4
  Opis: m4
  Materiał: N/A
  Kategoria: metal
  Tagi: metal

#3 (Dopasowanie: 85.1%)
  ID: m5
  Opis: m5
  Materiał: N/A
  Kategoria: metal
  Tagi: metal

#4 (Dopasowanie: 84.6%)
  ID: m7
  Opis: m7
  Materiał: N/A
  Kategoria: metal
  Tagi: metal

#5 (Dopasowanie: 84.5%)
  ID: m6
  Opis: m6
  Materiał: N/A
  Kategoria: metal
  Tagi: metal




In [ ]:
print("\n" + "="*70)
print("TEST 3: Losowy obraz")
print("="*70)

from pathlib import Path
import random
exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
candidates = [p for p in Path("obrazy/metal").rglob("*") if p.suffix.lower() in exts]

if len(candidates) == 0:
    print("Brak obrazów")
else:
    # Losowy obrazek
    query_img = str(random.choice(candidates))
    print(f"Obraz: {query_img}")
    # Szukaj bez filtra
    result3 = rag_search_images(query_img, top_k=5)
    print(f"\n{result3['report']}")


TEST 3: Image-only – inny obraz
Zapytanie (obraz): obrazy\metal\m1.jpg

Raport:
RAPORT WYSZUKIWANIA
Zapytanie: m1.jpg
Znaleziono: 5 części


#1 (Dopasowanie: 100.0%)
  ID: m1
  Opis: m1
  Materiał: N/A
  Kategoria: metal
  Tagi: metal

#2 (Dopasowanie: 81.9%)
  ID: m7
  Opis: m7
  Materiał: N/A
  Kategoria: metal
  Tagi: metal

#3 (Dopasowanie: 81.6%)
  ID: m2
  Opis: m2
  Materiał: N/A
  Kategoria: metal
  Tagi: metal

#4 (Dopasowanie: 80.8%)
  ID: m6
  Opis: m6
  Materiał: N/A
  Kategoria: metal
  Tagi: metal

#5 (Dopasowanie: 80.5%)
  ID: m4
  Opis: m4
  Materiał: N/A
  Kategoria: metal
  Tagi: metal




## 10. Statystyki i Podsumowanie

In [ ]:
with SessionLocal() as db:
    # Liczba części
    total_parts = db.query(PartDB).count()
    
    # Po kategoriach
    by_category = {}
    for part in db.query(PartDB).all():
        cat = part.category or "unknown"
        by_category[cat] = by_category.get(cat, 0) + 1
    
    # Logi wyszukiwań
    search_logs = db.query(SearchLog).all()
    
    print("\n" + "="*70)
    print("STATYSTYKI")
    print("="*70)
    print(f"Części: {total_parts}")
    print(f"\nPo kategoriach:")
    for cat, count in by_category.items():
        print(f"  - {cat}: {count}")
    print(f"\nWyszukiwań: {len(search_logs)}")
    
    if search_logs:
        print(f"\nOstatnie:")
        for log in search_logs[-5:]:
            print(f"  - {log.query_type}: '{log.query}' → {log.results_count}")


STATYSTYKI BAZY DANYCH
Łączna liczba części: 7

Podzielenie po kategoriach:
  - metal: 7

Liczba przeprowadzonych wyszukiwań: 12

Ostatnie wyszukiwania:
  - image: 'm2.webp' → 5 wyników
  - image: 'm5.webp' → 5 wyników
  - image: 'm1.jpg' → 5 wyników
  - image: 'm2.webp' → 5 wyników
  - image: 'm1.jpg' → 5 wyników


## 11. Zero-shot klasyfikacja części (CLIP)
Ten blok dodaje klasyfikację obrazów do etykiet (np. „łożysko”, „śruba”) bez trenowania — porównujemy embedding obrazu z embeddingami opisów etykiet.

In [ ]:
from typing import Sequence
import numpy as np

def get_text_embedding(texts: Sequence[str]) -> np.ndarray:
    """Generuj embeddingi dla tekstu używając modelu CLIP.
    
    Funkcja ta:
    - Przyjmuje listę tekstów (promptów)
    - Używa modelu sentence-transformers (CLIP) do kodowania tekstu
    - Normalizuje embeddingi (normalizacja L2)
    - Zwraca macierz numpy z embeddingami
    
    Args:
        texts: Lista tekstów do zakodowania
    
    Returns:
        Macierz numpy o kształcie (len(texts), embedding_dim)
    """
    embs = embedding_model.encode(list(texts), normalize_embeddings=True)
    return np.array(embs)


def softmax(x: np.ndarray, temperature: float = 0.01) -> np.ndarray:
    """Softmax z regulacją temperatury dla konwersji podobieństw na prawdopodobieństwa.
    
    Niska temperatura (0.01) sprawia, że rozkład jest bardziej zdecydowany -
    najwyższe podobieństwa dostają prawie całe prawdopodobieństwo.
    
    Args:
        x: Tablica podobieństw
        temperature: Parametr temperatury (niższa = bardziej zdecydowany)
    
    Returns:
        Tablica prawdopodobieństw (suma = 1.0)
    """
    x = x / temperature
    x = x - x.max()
    e = np.exp(x)
    return e / (e.sum() + 1e-9)


def classify_image_zero_shot(image_path: str,
                             labels_pl: Sequence[str],
                             template: str = "zdjęcie części mechanicznej: {}",
                             top_k: int = 5):
    """Klasyfikacja zero-shot CLIP.
    
    Proces klasyfikacji:
    1. Tworzy embedding obrazu zapytania
    2. Tworzy embeddingi tekstowe dla każdej etykiety (używając szablonu)
    3. Oblicza podobieństwo kosinusowe między embeddingiem obrazu a embeddingami tekstu
    4. Konwertuje podobieństwa na prawdopodobieństwa używając softmax z niską temperaturą
    5. Zwraca top_k najbardziej prawdopodobnych etykiet
    
    Przykład: Dla etykiety "łożysko" tworzy prompt "zdjęcie części mechanicznej: łożysko"
    
    Args:
        image_path: Ścieżka do obrazu do sklasyfikowania
        labels_pl: Lista etykiet tekstowych (np. ["łożysko", "śruba", "nakrętka"])
        template: Szablon dla promptów tekstowych (domyślnie po polsku)
        top_k: Liczba najlepszych wyników do zwrócenia
    
    Returns:
        Lista krotek (etykieta, prawdopodobieństwo, podobieństwo_surowe) posortowana malejąco
    """
    img_emb = get_image_embedding(image_path)
    if img_emb is None:
        return []

    prompts = [template.format(lbl) for lbl in labels_pl]
    txt_embs = get_text_embedding(prompts)

    img = np.array(img_emb)
    sims = txt_embs @ img
    probs = softmax(sims, temperature=0.02)

    ranked = sorted(zip(labels_pl, probs.tolist(), sims.tolist()), key=lambda t: t[1], reverse=True)
    return [(lbl, prob, sim) for lbl, prob, sim in ranked[:top_k]]

In [ ]:
from pathlib import Path

labels = [
    "łożysko", "śruba", "nakrętka", "podkładka", "wał",
    "sprężyna", "koło zębate", "zawór"
]

exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
candidates = [p for p in Path("obrazy/metal").rglob("*") if p.suffix.lower() in exts]

if not candidates:
    print("Brak obrazów")
else:
    # Test klasyfikacji na pierwszym obrazie z folderu
    test_img = str(candidates[0])
    print(f"Test: {test_img}")
    ranked = classify_image_zero_shot(test_img, labels, template="zdjęcie części metalowej: {}", top_k=5)
    for i, (lbl, prob, sim) in enumerate(ranked, 1):
        print(f"{i}. {lbl:<15} p={prob:0.3f}")

Klasyfikacja obrazu: obrazy\metal\m1.jpg

 1. zawór            p=0.197  sim=0.215
 2. śruba            p=0.146  sim=0.209
 3. sprężyna         p=0.133  sim=0.208
 4. nakrętka         p=0.124  sim=0.206
 5. wał              p=0.123  sim=0.206
 1. zawór            p=0.197  sim=0.215
 2. śruba            p=0.146  sim=0.209
 3. sprężyna         p=0.133  sim=0.208
 4. nakrętka         p=0.124  sim=0.206
 5. wał              p=0.123  sim=0.206


## 12. Klasyfikacja wszystkich obrazów w folderze i eksport do CSV
Ta sekcja:
- klasyfikuje każdy obraz z `obrazy/metal` do etykiet (np. „łożysko”, „śruba”…),
- zapisuje wyniki do `classification_results.csv`,
- aktualizuje tagi w bazie (`pred:<etykieta>`).

In [ ]:
from pathlib import Path
import csv

labels_all = [
    "łożysko", "śruba", "nakrętka", "podkładka", "wał", "sprężyna", "koło zębate", "zawór"
]

exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
images = [p for p in Path("obrazy/metal").rglob("*") if p.suffix.lower() in exts]

if not images:
    print("Brak obrazów w obrazy/metal")
else:
    rows = []
    with SessionLocal() as db:
        for p in images:
            img_path = str(p.as_posix())
            # Klasyfikuj obraz do top 3 etykiet
            ranked = classify_image_zero_shot(img_path, labels_all, template="zdjęcie części metalowej: {}", top_k=3)
            if not ranked:
                continue
            top1_label, top1_prob, _ = ranked[0]
            topk_str = "; ".join([f"{lbl}:{prob:0.3f}" for (lbl, prob, sim) in ranked])

            # Przygotuj wiersz CSV z wynikami klasyfikacji
            rows.append({
                "image_path": img_path,
                "top1_label": top1_label,
                "top1_prob": f"{top1_prob:0.3f}",
                "topk": topk_str
            })

            # Aktualizuj tagi w bazie danych - dodaj predykcję jako tag
            part = db.query(PartDB).filter(PartDB.image_path == img_path).first()
            if part is not None:
                tags = part.tags or []
                pred_tag = f"pred:{top1_label}"
                if pred_tag not in tags:
                    tags.append(pred_tag)
                    part.tags = tags
                    db.add(part)
                    # Zsynchronizuj z ChromaDB jeśli używane
                    try:
                        sync_part_to_chroma(part)
                    except Exception as e:
                        print(f"⚠️ Sync error: {e}")
        db.commit()

    # Zapisz wyniki do pliku CSV
    out_csv = Path("classification_results.csv")
    with out_csv.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["image_path", "top1_label", "top1_prob", "topk"])
        writer.writeheader()
        writer.writerows(rows)
    print(f"✓ Wyniki: {out_csv.resolve()} ({len(rows)} wierszy)")

⚠️ Chroma sync error for m1: name 'sync_part_to_chroma' is not defined

⚠️ Chroma sync error for m2: name 'sync_part_to_chroma' is not defined
⚠️ Chroma sync error for m2: name 'sync_part_to_chroma' is not defined
⚠️ Chroma sync error for m3: name 'sync_part_to_chroma' is not defined
⚠️ Chroma sync error for m3: name 'sync_part_to_chroma' is not defined
⚠️ Chroma sync error for m4: name 'sync_part_to_chroma' is not defined
⚠️ Chroma sync error for m4: name 'sync_part_to_chroma' is not defined
⚠️ Chroma sync error for m5: name 'sync_part_to_chroma' is not defined
⚠️ Chroma sync error for m5: name 'sync_part_to_chroma' is not defined
⚠️ Chroma sync error for m6: name 'sync_part_to_chroma' is not defined
⚠️ Chroma sync error for m7: name 'sync_part_to_chroma' is not defined
✓ Zapisano wyniki klasyfikacji: C:\Users\hubik\Desktop\vision_db\Vision_Picture_RAG\classification_results.csv (7 rekordów)
⚠️ Chroma sync error for m6: name 'sync_part_to_chroma' is not defined
⚠️ Chroma sync error fo

## 13. Auto-download obrazów z internetu dla każdej klasy
Automatyczne pobieranie i zapisywanie obrazów z Bing Image Search dla każdej etykiety.

In [ ]:
%pip install -q bing-image-downloader

from bing_image_downloader import downloader
from pathlib import Path
import shutil


def download_images_for_labels(labels: list, base_dir: Path = Path("obrazy/metal"), images_per_label: int = 10):
    """Pobierz obrazy z Bing Image Search dla każdej etykiety"""
    base_dir = Path(base_dir)
    base_dir.mkdir(parents=True, exist_ok=True)

    for label in labels:
        label_dir = base_dir / label
        label_dir.mkdir(parents=True, exist_ok=True)

        existing = []
        for ext in ("*.jpg", "*.jpeg", "*.png", "*.webp", "*.bmp"):
            existing.extend(label_dir.glob(ext))
        if len(existing) >= images_per_label:
            print(f"✓ '{label}' — {len(existing)} już istnieje")
            continue

        print(f"⬇ {label}...")
        try:
            query = f"{label} część mechaniczna"
            downloader.download(
                query,
                limit=images_per_label,
                output_dir="Dataset",
                adult_filter_off=True,
                force_replace=False,
                timeout=60,
                verbose=False,
            )

            dataset_root = Path("Dataset")
            dataset_dir = dataset_root / query
            if not dataset_dir.exists():
                candidates = [d for d in dataset_root.iterdir() if d.is_dir() and label.lower() in d.name.lower()]
                if candidates:
                    dataset_dir = max(candidates, key=lambda p: p.stat().st_mtime)

            moved = 0
            if dataset_dir.exists():
                for img_file in dataset_dir.glob("**/*"):
                    if img_file.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}:
                        dest = label_dir / f"{label}_{img_file.name}"
                        try:
                            shutil.move(str(img_file), str(dest))
                            moved += 1
                        except Exception:
                            img_file.rename(dest)
                            moved += 1
                try:
                    for p in sorted(dataset_dir.glob("**/*"), reverse=True):
                        if p.is_file():
                            p.unlink(missing_ok=True)
                        elif p.is_dir():
                            p.rmdir()
                    dataset_dir.rmdir()
                except Exception:
                    pass

            print(f"  ✓ +{moved} (razem: {len(list(label_dir.glob('*.*')))})")
        except Exception as e:
            print(f"  ✗ {label}: {e}")

    print(f"\n✓ Gotowe: {base_dir.resolve()}")

labels_to_download = [
    "łożysko kulkowe",
    "śruba heksagonalna",
    "nakrętka",
    "wał stalowy",
    "sprężyna",
]

print("Pobieranie z Bing...")
download_images_for_labels(labels_to_download, base_dir=Path("obrazy/metal"), images_per_label=5)

Note: you may need to restart the kernel to use updated packages.
Startuje pobieranie obrazów z Bing...
⬇ Pobieranie obrazów dla 'łożysko kulkowe'...
  ✗ Błąd przy 'łożysko kulkowe': download() got an unexpected keyword argument 'keyword'
⬇ Pobieranie obrazów dla 'śruba heksagonalna'...
  ✗ Błąd przy 'śruba heksagonalna': download() got an unexpected keyword argument 'keyword'
⬇ Pobieranie obrazów dla 'nakrętka'...
  ✗ Błąd przy 'nakrętka': download() got an unexpected keyword argument 'keyword'
⬇ Pobieranie obrazów dla 'wał stalowy'...
  ✗ Błąd przy 'wał stalowy': download() got an unexpected keyword argument 'keyword'
⬇ Pobieranie obrazów dla 'sprężyna'...
  ✗ Błąd przy 'sprężyna': download() got an unexpected keyword argument 'keyword'

✓ Pobieranie ukończone. Obrazy w: C:\Users\hubik\Desktop\vision_db\Vision_Picture_RAG\obrazy\metal



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 14. Automatyczne rozpoznawanie i opisywanie części za pomocą Vision API
Generuje dokładne opisy części metalowych używając zewnętrznego API (np. OpenAI GPT-4 Vision lub Google Gemini Vision).

In [ ]:
def describe_metal_part_clip(image_path: str, language: str = "pl") -> str:
    """Opisz część metalową porównując embedding obrazu z embeddingami szablonów opisowych (CLIP).
    
    Funkcja ta implementuje zero-shot image-to-text używając CLIP:
    1. Tworzy embedding obrazu wejściowego
    2. Porównuje go z embeddingami 40+ predefiniowanych opisów części metalowych
    3. Zwraca top 3 najbardziej podobne opisy wraz z poziomem pewności
    4. Formatuje wynik jako czytelny raport z nazwą i opisem części
    
    Dostępne języki: "pl" (polski), "en" (angielski)
    
    Args:
        image_path: Ścieżka do obrazu części
        language: Język opisów ("pl" lub "en")
    
    Returns:
        Sformatowany tekst z identyfikacją części i poziomem pewności
    """
    
    descriptions_pl = [
        "łożysko kulkowe - okrągły element z kulkami stalowymi do zmniejszania tarcia w ruchu obrotowym",
        "łożysko toczne - precyzyjny element mechaniczny z wewnętrznym i zewnętrznym pierścieniem",
        "łożysko ślizgowe - tuleją z gładką powierzchnią do łagodzenia tarcia",
        
        "śruba heksagonalna - element złączny z sześciokątną główką i gwintem metrycznym",
        "śruba imbusowa - śruba z wgłębieniem sześciokątnym w główce (klucz imbusowy)",
        "śruba z łbem stożkowym - śruba wpuszczana z płaskim łbem",
        "śruba oczkowa - śruba z pierścieniem zamiast główki do zawieszania",
        "wkręt samogwintujący - element z ostrym gwintem do wciskania w metal",
        
        "nakrętka sześciokątna - element gwintowany z sześciokątnym obrysem do dokręcania śrub",
        "nakrętka samohamowna - nakrętka z plastikowym wkładem zabezpieczającym przed odkręceniem",
        "nakrętka koronowa - nakrętka z nacięciami do zabezpieczenia zawleczką",
        "nakrętka skrzydełkowa - nakrętka z wystającymi skrzydełkami do ręcznego dokręcania",
        
        "podkładka płaska - okrągła tarcza stalowa do równomiernego rozłożenia nacisku",
        "podkładka sprężysta - falista podkładka zabezpieczająca przed samoistnym odkręceniem",
        "podkładka Grower'a - rozcięta podkładka sprężynująca zabezpieczająca połączenie",
        
        "wał stalowy - cylindryczny element przenoszący moment obrotowy",
        "wał z wpustem - wał z rowkiem do osadzania kół zębatych lub kół pasowych",
        "oś stalowa - wał nieobrotowy służący jako podpora dla kół lub rolek",
        
        "sprężyna śrubowa - zwój ze stali sprężynowej do magazynowania energii mechanicznej",
        "sprężyna rozciągana - sprężyna z haczykami do pracy na rozciąganie",
        "sprężyna naciskowa - sprężyna pracująca na ściskanie z cylindrycznym kształtem",
        "sprężyna talerzowa - płaska sprężyna w kształcie tarczy z wypukłością",
        
        "koło zębate - tarcza z uzębieniem na obwodzie do przekazywania napędu",
        "koło zębate walcowe - koło z prostymi zębami równoległymi do osi",
        "zębatka - listwa z uzębieniem liniowym współpracująca z kołem zębatym",
        "pastorek - małe koło zębate napędzające większe koło",
        
        "zawór kulowy - element z obrotową kulą z otworem do regulacji przepływu",
        "zawór zwrotny - element przepuszczający płyn tylko w jednym kierunku",
        "złączka gwintowana - element z gwintem wewnętrznym i zewnętrznym do łączenia rur",
        
        "sworzeń walcowy - gładki cylindryczny element do mocowania lub łączenia części",
        "sworzeń gwintowany - pręt stalowy z gwintem na końcach",
        "zawleczka - drut stalowy wygięty w kształt szpilki zabezpieczającej przed wysunięciem",
        
        "kołek walcowy - precyzyjny cylindryczny element do pozycjonowania części",
        "kołek stożkowy - element z lekkim stożkiem do samoczynnego zaciskania",
        "nit - trwały łącznik odkształcany plastycznie przy montażu",
        
        "tuleja dystansowa - cylindryczna tuleja do zachowania odstępu między częściami",
        "tuleja gwintowana - element z gwintem wewnętrznym do przedłużania połączeń",
        "prowadnica liniowa - profil stalowy do precyzyjnego ruchu liniowego",
        
        "obudowa metalowa - pojemnik lub osłona z blachy stalowej",
        "uchwyt metalowy - element mocujący lub trzymający wykonany ze stali",
        "płytka montażowa - płaska tarcza stalowa z otworami montażowymi"
    ]
    
    descriptions_en = [
        "ball bearing - circular element with steel balls to reduce friction in rotary motion",
        "rolling bearing - precision mechanical element with inner and outer rings",
        "hexagonal bolt - fastener with six-sided head and metric thread",
        "socket head cap screw - bolt with hexagonal socket for Allen key",
        "hexagonal nut - threaded element with hexagonal shape for tightening bolts",
        "flat washer - round steel disc to evenly distribute pressure",
        "steel shaft - cylindrical element for transmitting torque",
        "helical spring - coil of spring steel for storing mechanical energy",
        "spur gear - disc with teeth on perimeter for power transmission",
        "ball valve - element with rotating ball with hole for flow control"
    ]
    
    descriptions = descriptions_pl if language == "pl" else descriptions_en
    
    img_emb = get_image_embedding(image_path)
    txt_embs = embedding_model.encode(descriptions, normalize_embeddings=True)
    similarities = txt_embs @ np.array(img_emb)
    
    ranked_indices = np.argsort(similarities)[::-1]
    top_descriptions = []
    top_scores = []
    for idx in ranked_indices[:3]:
        top_descriptions.append(descriptions[idx])
        top_scores.append(float(similarities[idx]))
    
    result = f"**OPIS CZĘŚCI (CLIP)**\n\n**Top identyfikacje:**\n\n"
    for i, (desc, score) in enumerate(zip(top_descriptions, top_scores), 1):
        result += f"{i}. {desc}\n   Pewność: {score*100:.1f}%\n\n"
    
    main_description = top_descriptions[0].split(" - ")[0]
    full_description = top_descriptions[0]
    
    result += f"**Nazwa:** {main_description}\n"
    result += f"**Opis:** {full_description}\n"
    
    return result


def update_part_descriptions_clip(db: Session, limit: int = None):
    """Aktualizuj opisy części w bazie danych używając automatycznej identyfikacji CLIP.
    
    Proces aktualizacji:
    1. Pobiera wszystkie części z bazy (lub limit pierwszych)
    2. Dla każdej części generuje opis używając describe_metal_part_clip()
    3. Wyciąga nazwę części z wygenerowanego opisu
    4. Aktualizuje pole description w bazie danych
    5. Synchronizuje zmiany z ChromaDB (jeśli używane)
    
    Ta funkcja pozwala na automatyczne wzbogacenie metadanych części
    bez ręcznego wprowadzania opisów.
    
    Args:
        db: Sesja SQLAlchemy do bazy danych
        limit: Opcjonalny limit liczby części do przetworzenia (None = wszystkie)
    """
    parts = db.query(PartDB).all()
    if limit:
        parts = parts[:limit]
    
    updated = 0
    for part in parts:
        if not os.path.exists(part.image_path):
            print(f"⚠️  Brak: {part.image_path}")
            continue
        
        print(f"🔍 {Path(part.image_path).name}...")
        description = describe_metal_part_clip(part.image_path, language="pl")
        
        lines = description.split("\n")
        for line in lines:
            if line.startswith("**Nazwa:**"):
                part.description = line.replace("**Nazwa:**", "").strip()
                break
        
        db.add(part)
        try:
            sync_part_to_chroma(part)
        except Exception as e:
            print(f"⚠️ Chroma error: {e}")
        
        updated += 1
        print(f"✓ {part.part_id}")
    
    db.commit()
    print(f"\n✓ Zaktualizowano {updated}/{len(parts)}")


print("✓ CLIP (bezpłatne opisy)")
print("✓ Brak limitów API, działa offline")

✓ Funkcje CLIP do opisywania części załadowane (BEZPŁATNE)
✓ Brak limitów API, działa offline po załadowaniu modelu CLIP


### Test pojedynczego obrazu

In [ ]:
test_images = list(Path("obrazy/metal").rglob("*.jpg"))[:1] + list(Path("obrazy/metal").rglob("*.webp"))[:1]

if test_images:
    test_image = str(test_images[0])
    print(f"📸 Test z: {Path(test_image).name}\n")
    
    description = describe_metal_part_clip(test_image, language="pl")
    print("=" * 80)
    print(description)
    print("=" * 80)
else:
    print("⚠️ Brak obrazów")

📸 Testuję z obrazem: m1.jpg

**OPIS CZĘŚCI (wygenerowany przez CLIP - bezpłatnie)**

**Najbardziej prawdopodobne identyfikacje:**

1. sworzeń walcowy - gładki cylindryczny element do mocowania lub łączenia części
   Pewność: 22.7%

2. kołek walcowy - precyzyjny cylindryczny element do pozycjonowania części
   Pewność: 22.0%

3. wał stalowy - cylindryczny element przenoszący moment obrotowy
   Pewność: 21.8%

**Rekomendowana nazwa:** sworzeń walcowy
**Pełny opis:** sworzeń walcowy - gładki cylindryczny element do mocowania lub łączenia części

**OPIS CZĘŚCI (wygenerowany przez CLIP - bezpłatnie)**

**Najbardziej prawdopodobne identyfikacje:**

1. sworzeń walcowy - gładki cylindryczny element do mocowania lub łączenia części
   Pewność: 22.7%

2. kołek walcowy - precyzyjny cylindryczny element do pozycjonowania części
   Pewność: 22.0%

3. wał stalowy - cylindryczny element przenoszący moment obrotowy
   Pewność: 21.8%

**Rekomendowana nazwa:** sworzeń walcowy
**Pełny opis:** sworzeń wal

### Aktualizacja opisów dla wszystkich części w bazie

In [ ]:
# Aktualizuj opisy części (offline, bezpłatne)
# update_part_descriptions_clip(db, limit=5)      # test: 5 obrazów
# update_part_descriptions_clip(db, limit=None)   # wszystkie

print("✓ CLIP - bezpłatne opisy bez limitów API")
print("✓ Działa offline po załadowaniu modelu")
print("✓ Porównuje obraz z 40+ szablonami części")

✅ CLIP - CAŁKOWICIE BEZPŁATNE ROZWIĄZANIE
✓ Brak limitów API
✓ Działa offline (bez internetu po załadowaniu modelu)
✓ Opisuje części porównując obraz z 40+ szablonami tekstowymi

⚠️ Odkomentuj jedną z opcji powyżej, aby uruchomić aktualizację


In [ ]:
from typing import Any

def sync_part_to_chroma(part: PartDB) -> None:
    """Synchronizuj dane części między SQLite a ChromaDB.
    
    Funkcja ta zapewnia spójność danych między:
    - Relacyjną bazą SQLite (metadane, ścieżki)
    - Wektową bazą ChromaDB (embeddingi do wyszukiwania)
    
    Aktualizuje lub wstawia dokument w ChromaDB z aktualnymi metadanymi części.
    Używane po każdej zmianie danych części (opis, tagi, kategoria).
    
    Args:
        part: Obiekt PartDB z bazy danych do zsynchronizowania
    """
    if not USE_CHROMA:
        return
    try:
        md = {
            "part_id": part.part_id,
            "description": part.description or "",
            "material": part.material or "N/A",
            "category": part.category or "metal",
            "image_path": part.image_path or "",
            "tags": ",".join(part.tags or [])
        }
        if hasattr(chroma_collection, "update"):
            chroma_collection.update(
                ids=[part.part_id],
                documents=[part.description or ""],
                metadatas=[md]
            )
        else:
            chroma_collection.upsert(
                ids=[part.part_id],
                documents=[part.description or ""],
                metadatas=[md]
            )
    except Exception as e:
        print(f"⚠️ Sync failed: {e}")

print("✓ Sync załadowany")

✓ Funkcja sync_part_to_chroma załadowana
